# Santander Value Prediction — Decision Tree Experiments

> **Stage:** First feature-based model and controlled regularization study · **Status:** Exploratory experiments complete

**Question:** Can a Decision Tree exploit anonymized customer features to beat a constant log-mean baseline?

**Approach:** Begin with an interpretable, low-complexity model, evaluate RMSLE, then study generalization under a **common five-fold cross-validation (CV)** setup.

The original 80/20 holdout was used for early exploratory feedback and has been inspected. Hyperparameter experiments below are run **only on the 80% training portion**. Reported scores are not final unbiased test results.


## 1 · Data and reproducible evaluation setup

Log-transforming the non-negative target aligns squared-error learning with the RMSLE evaluation scale. `neg_root_mean_squared_error` in log space is used because scikit-learn defines scores as "higher is better".

All models below use **the same folds** so comparisons are on matching observations.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import (
    KFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.tree import DecisionTreeRegressor


In [2]:
dataset = fetch_openml(
    data_id=42572,
    target_column="target",
    as_frame=True,
)

X = dataset.data
y = dataset.target

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (4459, 4991)
y shape: (4459,)


In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)


X_train: (3567, 4991)
X_val: (892, 4991)
y_train: (3567,)
y_val: (892,)


In [4]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
y_train_log = np.log1p(y_train)


## 2 · Initial Decision Tree (early holdout experiment)

**Why Decision Tree:** Handles non-linear splits and does not require feature scaling. `max_depth=5` and `min_samples_leaf=20` were *initial* restrictions to reduce sensitivity to small sample groups, not claimed optima.

**Procedure:** Fit on `X_train` and log-transformed targets; inverse-transform predictions with `expm1` before reporting holdout RMSLE.


In [5]:
tree_model = DecisionTreeRegressor(
    max_depth=5,
    min_samples_leaf=20,
    random_state=42,
)

tree_model.fit(X_train, y_train_log)


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with `

In [6]:
train_pred = np.expm1(tree_model.predict(X_train))
val_pred = np.expm1(tree_model.predict(X_val))

train_rmsle = root_mean_squared_log_error(y_train, train_pred)
val_rmsle = root_mean_squared_log_error(y_val, val_pred)

print(f"Train RMSLE: {train_rmsle:.4f}")
print(f"Validation RMSLE: {val_rmsle:.4f}")


Train RMSLE: 1.5782
Validation RMSLE: 1.5862


**Recorded holdout results:** train RMSLE **1.5782**, validation RMSLE **1.5862**. The tree improves over the log-mean holdout baseline (1.6963), with no evidence here of a very large holdout/train gap.

**Decision:** A single holdout may flatter or penalize a model. Switch to five-fold CV on the training subset before comparing configurations.


## 3 · Five-fold CV of the initial tree

`cross_val_score` clones and refits the tree on each CV training fold. Its `test_score` is the **validation part of that fold**; the reserved `X_val` is not included in this CV call.


In [7]:
cv_scores = cross_val_score(
    tree_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error",
)

cv_rmsle = -cv_scores
print("Fold RMSLE:", cv_rmsle)
print(f"Mean CV RMSLE: {cv_rmsle.mean():.4f}")
print(f"Std CV RMSLE: {cv_rmsle.std():.4f}")


Fold RMSLE: [1.67861101 1.70546742 1.75043304 1.66853608 1.66438486]
Mean CV RMSLE: 1.6935
Std CV RMSLE: 0.0319


**Finding:** Mean CV RMSLE **1.6935** (fold-to-fold standard deviation **0.0319**), higher than the original holdout result. Modest SD means scores are similar across these folds; it does **not** prove balanced classes (this is regression) or perfect generalization.


## 4 · Fair, paired baseline comparison

**Why:** Compare the tree to a log-mean constant baseline on **the exact same folds**. `DummyRegressor(strategy="mean")` trained on `log1p(y)` produces the log-mean prediction within each CV training partition.


In [8]:
baseline_model = DummyRegressor(strategy="mean")

baseline_cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error",
)

baseline_cv_rmsle = -baseline_cv_scores
print(f"Baseline CV RMSLE: {baseline_cv_rmsle.mean():.4f}")
print(f"Decision Tree CV RMSLE: {cv_rmsle.mean():.4f}")


Baseline CV RMSLE: 1.7644
Decision Tree CV RMSLE: 1.6935


In [9]:
comparison = pd.DataFrame({
    "Log-Mean Baseline": baseline_cv_rmsle,
    "Decision Tree": cv_rmsle,
})
comparison["Improvement"] = (
    comparison["Log-Mean Baseline"] - comparison["Decision Tree"]
)

display(comparison)


,Log-Mean Baseline,Decision Tree,Improvement
0,1.780768,1.678611,0.102157
1,1.757487,1.705467,0.052019
2,1.798021,1.750433,0.047588
3,1.737518,1.668536,0.068982
4,1.748384,1.664385,0.083999


**Findings:** Baseline mean CV RMSLE **1.7644**, initial tree **1.6935**. Tree wins on all five matched folds; the average CV error is about **4.0%** lower.

**Decision:** Explore tree complexity and regularization. These five folds are used repeatedly for model selection, so the chosen model's best CV score is likely a somewhat optimistic estimate of new-data performance.


## 5 · Complexity study: maximum depth

**Hypothesis:** Increased depth reduces training error, but at some point no longer improves performance on unseen folds.

**Controlled experiment:** Vary `max_depth` ∈ {2, 5, 10, 15, 20}; keep `min_samples_leaf=20`, `random_state=42`, and CV folds fixed.

The **generalization gap** is the average CV validation error minus the average training-fold error. A growing gap is informative, but not a sufficient criterion on its own for rejecting a model.


In [10]:
depth_results = []

for depth in [2, 5, 10, 15, 20]:
    model = DecisionTreeRegressor(
        max_depth=depth,
        min_samples_leaf=20,
        random_state=42,
    )
    scores = cross_validate(
        model,
        X_train,
        y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
    )
    train_error = -scores["train_score"].mean()
    validation_error = -scores["test_score"].mean()
    depth_results.append({
        "max_depth": depth,
        "train_rmsle": train_error,
        "cv_rmsle": validation_error,
        "gap": validation_error - train_error,
    })

depth_results_df = pd.DataFrame(depth_results)
display(depth_results_df)


,max_depth,train_rmsle,cv_rmsle,gap
0,2,1.681328,1.710894,0.029566
1,5,1.581771,1.693486,0.111715
2,10,1.453801,1.675131,0.221330
3,15,1.395167,1.674953,0.279786
4,20,1.370545,1.678306,0.307762


**Previously observed results for depths 10–20** (from earlier experiments; run the code above to regenerate the full table):

| Max depth | Train RMSLE | CV RMSLE | Gap |
|---|---:|---:|---:|
| 10 | 1.4538 | 1.6751 | 0.2213 |
| 15 | 1.3952 | 1.6750 | 0.2798 |
| 20 | 1.3705 | 1.6783 | 0.3078 |

**Interpretation:** Training error decreases with depth, but CV error stops improving materially beyond 10. Depth 15 is marginally best on these folds (difference ≈ 0.00018 from 10) and has a larger gap.

**Decision:** Use depth 10 as a simple reference while studying another regularization control. Depth 10 is *not* asserted to be globally optimal.


## 6 · Regularization study: minimum leaf size

**Hypothesis:** Tiny leaf groups can capture chance variation in the training set. Requiring more observations per leaf may improve generalization.

**Controlled experiment:** Fix `max_depth=10`; test leaf sizes 5, 20, and 50 using the same folds and metric.


In [11]:
leaf_results = []

for leaf_size in [5, 20, 50]:
    model = DecisionTreeRegressor(
        max_depth=10,
        min_samples_leaf=leaf_size,
        random_state=42,
    )
    scores = cross_validate(
        model,
        X_train,
        y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
    )
    train_error = -scores["train_score"].mean()
    validation_error = -scores["test_score"].mean()
    leaf_results.append({
        "min_samples_leaf": leaf_size,
        "train_rmsle": train_error,
        "cv_rmsle": validation_error,
        "gap": validation_error - train_error,
    })

leaf_results_df = pd.DataFrame(leaf_results)
display(leaf_results_df)


,min_samples_leaf,train_rmsle,cv_rmsle,gap
0,5,1.355656,1.734650,0.378994
1,20,1.453801,1.675131,0.221330
2,50,1.516316,1.645829,0.129513


**Recorded results (max_depth = 10):**

| Min leaf samples | Train RMSLE | Mean CV RMSLE | Gap |
|---|---:|---:|---:|
| 5 | **1.3557** | 1.7347 | 0.3790 |
| 20 | 1.4538 | 1.6751 | 0.2213 |
| **50** | 1.5163 | **1.6458** | **0.1295** |

**Interpretation:** Leaf size 5 fits training folds very closely, but has the worst CV error. Increasing leaf size to 50 increases training error yet decreases CV error, consistent with useful regularization.

**Decision:** Keep `max_depth=10`, `min_samples_leaf=50` as the *best configuration of the limited candidates tried*, not a globally optimized model.


## 7 · Summary, caveats, next steps

| Model / setup | Evaluation | RMSLE |
|---|---|---:|
| Log-mean baseline | Mean 5-fold CV | 1.7644 |
| Initial tree (depth 5, leaf 20) | Mean 5-fold CV | 1.6935 |
| Best tested tree (depth 10, leaf 50) | Mean 5-fold CV | **1.6458** |

**Relative mean-CV improvement vs baseline:** ≈ **6.72%**. This comparison uses the same training subset and five-fold evaluation design. It does **not** represent a Kaggle leaderboard score.

**Lessons:**
- Evaluate against a simple baseline, not training error alone.
- Use matched folds and consistent metrics.
- A bigger train–validation gap does not automatically mean a model is worse; inspect CV error as well.
- Hyperparameter selection on repeated CV folds introduces selection bias.

**Next:** Compare ensemble methods, examine candidate features, and define an honest final evaluation protocol. The early `X_val` was already inspected; avoid labeling it a never-seen final test set.
